
# Assignment 3 – Mushroom Classification

**Goal:** Build and compare multiple classification models, tune at least three models, and generate a Kaggle submission.

The notebook follows the peer-review rubric:
- Data types
- Descriptive statistics
- Missing values
- Duplicates
- Outliers
- 3+ visualizations with insights
- Numerical scaling and categorical encoding
- 7 classification models
- Hyperparameter tuning for 3 models
- Model-performance comparison
- Final Kaggle submission


In [ ]:

# Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from catboost import CatBoostClassifier

RANDOM_STATE = 42


In [ ]:

# Load the Kaggle files
import glob

def find_kaggle_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    return matches[0]

train_path = find_kaggle_file("train.csv")
test_path = find_kaggle_file("test.csv")
sample_path = find_kaggle_file("sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())


## 1. Data Types

In [ ]:

# Identify data types
dtype_table = pd.DataFrame({
    "Column": train_df.columns,
    "Data Type": train_df.dtypes.astype(str).values,
    "Unique Values": [train_df[c].nunique(dropna=False) for c in train_df.columns]
})

display(dtype_table)

print("\nNumerical columns:")
print(train_df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical columns:")
print(train_df.select_dtypes(exclude=np.number).columns.tolist())


## 2. Descriptive Statistics

In [ ]:

# Numerical descriptive statistics
numeric_cols_all = train_df.select_dtypes(include=np.number).columns.tolist()

stats = pd.DataFrame({
    "Minimum": train_df[numeric_cols_all].min(),
    "Maximum": train_df[numeric_cols_all].max(),
    "Mean": train_df[numeric_cols_all].mean(),
    "Median": train_df[numeric_cols_all].median()
})

display(stats)


## 3. Missing Values

In [ ]:

missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)

missing_table = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percentage": missing_pct
}).sort_values("Missing Count", ascending=False)

display(missing_table[missing_table["Missing Count"] > 0])

print("Total missing cells:", int(train_df.isnull().sum().sum()))

# We will impute missing numerical values with the median and
# missing categorical values with the most frequent category.
print("\nHandling plan:")
print("- Numerical features: median imputation")
print("- Categorical features: most-frequent imputation")


## 4. Duplicate Records

In [ ]:

duplicate_count = train_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    train_df = train_df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows found, so no rows were removed.")


## 5. Outlier Detection

In [ ]:

# IQR-based outlier counts for numerical columns.
# Identifier columns are excluded from the outlier decision.
id_like_cols = [c for c in ["ID", "mushroom_id"] if c in train_df.columns]
numeric_model_cols = [
    c for c in train_df.select_dtypes(include=np.number).columns
    if c not in id_like_cols
]

outlier_rows = []
for col in numeric_model_cols:
    s = train_df[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((s < lower) | (s > upper)).sum()
    outlier_rows.append([col, q1, q3, lower, upper, int(count)])

outlier_table = pd.DataFrame(
    outlier_rows,
    columns=["Column", "Q1", "Q3", "Lower Bound", "Upper Bound", "Outlier Count"]
)

display(outlier_table)

print(
    "\nDecision: numerical outliers are retained. "
    "The mushroom dataset contains discrete/categorical-style measurements, "
    "so deleting valid observations based only on the IQR rule could remove "
    "useful information. Scaling and tree-based models also reduce the need "
    "for aggressive outlier deletion."
)


## 6. Visualizations and Insights

In [ ]:

# Visualization 1: Target distribution
plt.figure(figsize=(6, 4))
train_df["class"].value_counts().plot(kind="bar")
plt.title("Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.show()

print("Insight: The two target classes are reasonably balanced, so accuracy is a useful primary metric.")


In [ ]:

# Visualization 2: Odor vs class
odor_class = pd.crosstab(train_df["odor"], train_df["class"])

odor_class.plot(kind="bar", figsize=(9, 5))
plt.title("Mushroom Odor vs Class")
plt.xlabel("Odor")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.legend(title="Class")
plt.tight_layout()
plt.show()

print(
    "Insight: Odor is highly informative. Several odor categories are strongly "
    "associated with a single class, making it an important predictive feature."
)


In [ ]:

# Visualization 3: Habitat vs class
habitat_class = pd.crosstab(train_df["habitat"], train_df["class"])

habitat_class.plot(kind="bar", figsize=(9, 5))
plt.title("Habitat vs Class")
plt.xlabel("Habitat")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.legend(title="Class")
plt.tight_layout()
plt.show()

print(
    "Insight: Class proportions vary across habitats. Habitat therefore provides "
    "additional information when combined with other mushroom characteristics."
)



## 7. Feature Preparation

`ID` and `mushroom_id` are treated as identifiers rather than biological features and are excluded from model training.

For the sklearn models:
- Missing numerical values → median
- Missing categorical values → most frequent category
- Numerical values → `StandardScaler`
- Categorical values → `OneHotEncoder(handle_unknown="ignore")`

CatBoost is handled separately because it can work directly with categorical columns.


In [ ]:

target = "class"
drop_cols = [c for c in ["ID", "mushroom_id"] if c in train_df.columns]

X = train_df.drop(columns=[target] + drop_cols).copy()
y = train_df[target].map({"e": 0, "p": 1})

X_test = test_df.drop(columns=drop_cols, errors="ignore").copy()

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("Features used:", len(X.columns))
print("Numerical features:", numeric_cols)
print("Categorical features:", categorical_cols)


In [ ]:

# Train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Common preprocessing pipeline
preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]),
        categorical_cols
    )
])

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])


## 8. Model Building – 7 Different Models

In [ ]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Gaussian Naive Bayes": GaussianNB(),
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, pred),
        "Precision": precision_score(y_valid, pred),
        "Recall": recall_score(y_valid, pred),
        "F1": f1_score(y_valid, pred)
    })
    trained_models[name] = pipe

# 7th model: CatBoost
X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()

for c in categorical_cols:
    X_train_cb[c] = X_train_cb[c].fillna("Missing").astype(str)
    X_valid_cb[c] = X_valid_cb[c].fillna("Missing").astype(str)

cat_indices = [X_train_cb.columns.get_loc(c) for c in categorical_cols]

cat_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.08,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False
)

cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_indices
)

cat_pred = cat_model.predict(X_valid_cb).astype(int).ravel()

results.append({
    "Model": "CatBoost",
    "Accuracy": accuracy_score(y_valid, cat_pred),
    "Precision": precision_score(y_valid, cat_pred),
    "Recall": recall_score(y_valid, cat_pred),
    "F1": f1_score(y_valid, cat_pred)
})

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(results_df)



## 9. Hyperparameter Tuning – 3 Models

Three tree-based models are tuned using cross-validation:
1. Decision Tree
2. Random Forest
3. Extra Trees

The tuned models are evaluated on the same held-out validation set for a fair comparison.


In [ ]:

# Decision Tree tuning
dt_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

dt_grid = {
    "model__max_depth": [None, 5, 8, 12, 16],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

dt_search = GridSearchCV(
    dt_pipe, dt_grid, cv=3, scoring="accuracy", n_jobs=-1
)
dt_search.fit(X_train, y_train)

print("Best Decision Tree parameters:")
print(dt_search.best_params_)
print("Best CV accuracy:", dt_search.best_score_)


In [ ]:

# Random Forest tuning
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2]
}

rf_search = GridSearchCV(
    rf_pipe, rf_grid, cv=3, scoring="accuracy", n_jobs=-1
)
rf_search.fit(X_train, y_train)

print("Best Random Forest parameters:")
print(rf_search.best_params_)
print("Best CV accuracy:", rf_search.best_score_)


In [ ]:

# Extra Trees tuning
et_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

et_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2]
}

et_search = GridSearchCV(
    et_pipe, et_grid, cv=3, scoring="accuracy", n_jobs=-1
)
et_search.fit(X_train, y_train)

print("Best Extra Trees parameters:")
print(et_search.best_params_)
print("Best CV accuracy:", et_search.best_score_)


## 10. Compare Tuned Models

In [ ]:

tuned_models = {
    "Tuned Decision Tree": dt_search.best_estimator_,
    "Tuned Random Forest": rf_search.best_estimator_,
    "Tuned Extra Trees": et_search.best_estimator_
}

tuned_results = []

for name, model in tuned_models.items():
    pred = model.predict(X_valid)
    tuned_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, pred),
        "Precision": precision_score(y_valid, pred),
        "Recall": recall_score(y_valid, pred),
        "F1": f1_score(y_valid, pred)
    })

tuned_results_df = pd.DataFrame(tuned_results)

all_results = pd.concat(
    [results_df, tuned_results_df],
    ignore_index=True
).sort_values("Accuracy", ascending=False).reset_index(drop=True)

display(all_results)



## 11. Final Model Selection

The validation results are used to select a high-performing final model.

CatBoost is particularly suitable here because the dataset contains many categorical features and missing categorical values. The final model is retrained on **all available training rows** before generating test predictions.


In [ ]:

# Final CatBoost training on all training data
X_full_cb = X.copy()
X_test_cb = X_test.copy()

for c in categorical_cols:
    X_full_cb[c] = X_full_cb[c].fillna("Missing").astype(str)
    X_test_cb[c] = X_test_cb[c].fillna("Missing").astype(str)

cat_indices_full = [X_full_cb.columns.get_loc(c) for c in categorical_cols]

final_model = CatBoostClassifier(
    iterations=700,
    depth=8,
    learning_rate=0.06,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False
)

final_model.fit(
    X_full_cb,
    y,
    cat_features=cat_indices_full
)

test_pred_numeric = final_model.predict(X_test_cb).astype(int).ravel()
test_pred = np.where(test_pred_numeric == 1, "p", "e")

print("Test predictions generated:", len(test_pred))
print(pd.Series(test_pred).value_counts())


## 12. Create Kaggle Submission

In [ ]:

# Build submission using the exact ID column from sample_submission
submission = sample_submission.copy()
submission["class"] = test_pred

# Keep exactly the columns expected by Kaggle
submission = submission[["ID", "class"]]

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully.")
display(submission.head())
print("Submission shape:", submission.shape)



## Final Takeaways

- The dataset contains a mixture of numerical and categorical variables.
- Missing values were identified and handled through imputation.
- No duplicate rows were found.
- IQR-based outliers were checked and retained because the variables are largely discrete and valid observations should not be removed solely from an IQR rule.
- Three visualizations were used to understand class balance and important feature relationships.
- Numerical features were scaled and categorical features were one-hot encoded for sklearn models.
- Seven different classification algorithms were evaluated.
- Three models were hyperparameter tuned with cross-validation.
- CatBoost was selected for the final submission because it performs strongly on mixed categorical data and handles missing categorical values naturally after explicit filling.
